In [8]:
# Cell 1: 필수 라이브러리 설치
# undetected-chromedriver와 selenium을 설치합니다.
%pip install undetected-chromedriver selenium
# OpenAI 라이브러리 설치 (이미 설치되어 있다면 스킵)
%pip install openai --quiet
# python-dotenv 설치 (.env 파일 로드를 위해 필요)
%pip install python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
# Cell 2: 라이브러리 import 및 설정
import sys
import os
import tempfile
import subprocess
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import json
import time
import re
import inspect
import textwrap
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요(True로 하면 상품 탐색이 안 됩니다)
NOTEBOOK_HEADLESS = False

# 검색할 키워드 입력 (여기를 수정하세요)
SEARCH_KEYWORD = "컴퓨터"  # 예시: "노트북", "무선 이어폰", "스마트폰" 등

# 최대 수집할 상품 개수
MAX_PRODUCTS = 30  # 테스트용 (비용 문제로 5로 설정)

# 검색 URL 생성
search_url = f"https://www.coupang.com/np/search?q={quote(SEARCH_KEYWORD)}"

print(f"🔍 검색 키워드: {SEARCH_KEYWORD}")
print(f"🔗 검색 URL: {search_url}")
print(f"📦 최대 수집 개수: {MAX_PRODUCTS}개")
print(f"📸 상세 이미지 추출: Cell 6에서 FETCH_DETAIL_IMAGES 설정 가능\n")


🔍 검색 키워드: 컴퓨터
🔗 검색 URL: https://www.coupang.com/np/search?q=%EC%BB%B4%ED%93%A8%ED%84%B0
📦 최대 수집 개수: 30개
📸 상세 이미지 추출: Cell 6에서 FETCH_DETAIL_IMAGES 설정 가능



In [16]:
# Cell 3: OpenAI Vision API 설정 및 이미지 분석 함수 정의
from openai import OpenAI
from openai import RateLimitError, APIError
import os
from dotenv import load_dotenv
from pathlib import Path

# 프로젝트 루트 디렉토리 찾기 (노트북 위치 기준)
# 노트북은 dev/crawling_tests/에 있으므로, 상위 2단계가 루트
notebook_dir = Path.cwd()  # 현재 노트북 실행 디렉토리
project_root = notebook_dir.parent.parent  # dev/crawling_tests -> dev -> Final-AI-Fork
env_path = project_root / ".env"

# .env 파일에서 환경 변수 로드
load_dotenv(env_path)

# OpenAI API 키 설정 (.env 파일에서 가져오기)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY가 설정되지 않았습니다.")
    print(f"   .env 파일 경로: {env_path}")
    print("   프로젝트 루트의 .env 파일에 OPENAI_API_KEY를 설정해주세요.")
    print("   예시: OPENAI_API_KEY=sk-your-api-key-here")
else:
    print("✅ OpenAI API 키가 설정되었습니다.")
    print(f"   .env 파일 경로: {env_path}")

# OpenAI 클라이언트 초기화
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# 이미지 분석을 위한 프롬프트 정의
EXTRACTION_PROMPT = """이 이미지는 쿠팡 상품의 상세 정보 이미지입니다.
이미지에서 다음 정보들을 추출해주세요:

1. **제품명/모델명**: 제품의 이름이나 모델 번호
2. **색상(COLOR)**: 사용 가능한 색상 옵션
3. **사이즈(SIZE)**: 사이즈 정보, 치수, 크기
4. **소재/재질**: 제품의 소재나 재질 정보
5. **제조사/브랜드**: 제조사, 브랜드, 판매자 정보
6. **세부 사양**: 무게, 용량, 규격 등 기타 사양
7. **기타 정보**: 세탁 방법, 주의사항, 인증 정보 등

결과를 JSON 형식으로 정리해주세요. 정보가 없는 항목은 null로 표시하세요.
예시:
{
    "product_name": "제품명",
    "colors": ["색상1", "색상2"],
    "sizes": ["S", "M", "L"],
    "material": "소재 정보",
    "manufacturer": "제조사",
    "specifications": {"무게": "100g", "용량": "500ml"},
    "other_info": ["세탁 방법", "주의사항"]
}
"""


def retry_with_backoff(max_retries=3, base_wait=2, rate_limit_wait=60):
    """
    OpenAI API 호출을 위한 재시도 데코레이터
    
    Args:
        max_retries: 최대 재시도 횟수 (기본값: 3)
        base_wait: 기본 대기 시간 (초, exponential backoff의 베이스)
        rate_limit_wait: Rate limit 에러 시 최소 대기 시간 (초)
    
    Returns:
        데코레이터 함수
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                
                except RateLimitError as e:
                    # Rate limiting 에러: 더 긴 대기 시간 필요
                    if attempt < max_retries - 1:
                        wait_time = rate_limit_wait + (2 ** attempt)  # 최소 60초 + exponential backoff
                        print(f"  ⚠️ Rate limit 도달. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                        time.sleep(wait_time)
                        continue
                    return {"error": f"Rate limit 에러: {str(e)}"}
                
                except APIError as e:
                    # API 에러 (500, 503 등): 재시도 가능
                    if attempt < max_retries - 1:
                        wait_time = base_wait ** attempt  # exponential backoff
                        print(f"  ⚠️ API 에러 발생. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                        time.sleep(wait_time)
                        continue
                    return {"error": f"API 에러: {str(e)}"}
                
                except Exception as e:
                    # 기타 에러: 재시도 가능한 경우만 재시도
                    if attempt < max_retries - 1:
                        # 네트워크 에러나 타임아웃 등은 재시도
                        error_str = str(e).lower()
                        if any(keyword in error_str for keyword in ["timeout", "connection", "network", "503", "500"]):
                            wait_time = base_wait ** attempt  # exponential backoff
                            print(f"  ⚠️ 일시적 에러 발생. {wait_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                            time.sleep(wait_time)
                            continue
                    return {"error": str(e)}
            
            return {"error": "최대 재시도 횟수 초과"}
        
        return wrapper
    return decorator

@retry_with_backoff(max_retries=3)
def analyze_product_image(image_url, prompt=EXTRACTION_PROMPT, max_tokens=1500):
    """
    OpenAI Vision API를 사용하여 상품 이미지에서 텍스트를 추출합니다.
    
    Args:
        image_url: 분석할 이미지 URL
        prompt: 추출 프롬프트
        max_tokens: 최대 응답 토큰 수
        
    Returns:
        str: 추출된 텍스트/정보
    """
    if not client:
        return {"error": "OpenAI API 키가 설정되지 않았습니다."}
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Vision 지원 모델 (gpt-4o, gpt-4-turbo 등도 가능)
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_url,
                            "detail": "high"  # 고해상도 분석
                        }
                    }
                ]
            }
        ],
        max_tokens=max_tokens,
        timeout=30  # 타임아웃 설정 (30초)
    )
    
    return response.choices[0].message.content


def analyze_multiple_images(image_urls, prompt=EXTRACTION_PROMPT, max_images=5):
    """
    여러 이미지를 분석하여 상품 정보를 추출합니다.
    
    Args:
        image_urls: 이미지 URL 리스트
        prompt: 추출 프롬프트
        max_images: 분석할 최대 이미지 수 (비용 절감)
        
    Returns:
        list: 각 이미지에서 추출된 정보 리스트
    """
    if not client:
        return [{"error": "OpenAI API 키가 설정되지 않았습니다."}]
    
    results = []
    images_to_analyze = image_urls[:max_images]  # 비용 절감을 위해 제한
    
    for idx, url in enumerate(images_to_analyze):
        print(f"  📷 이미지 {idx+1}/{len(images_to_analyze)} 분석 중...")
        result = analyze_product_image(url, prompt)
        results.append({
            "image_url": url,
            "extracted_info": result
        })
        time.sleep(1)  # API 요청 간 딜레이
    
    return results


@retry_with_backoff(max_retries=3)
def analyze_product_images_combined(image_urls, max_images=3):
    """
    여러 이미지를 하나의 요청으로 분석합니다 (비용 효율적).
    
    Args:
        image_urls: 이미지 URL 리스트
        max_images: 분석할 최대 이미지 수
        
    Returns:
        str: 통합된 분석 결과
    """
    if not client:
        return {"error": "OpenAI API 키가 설정되지 않았습니다."}
    
    if not image_urls:
        return {"error": "분석할 이미지가 없습니다."}
    
    # 첫 번째, 중간, 마지막 이미지 선택 (다양한 정보 수집을 위해)
    total_images = len(image_urls)
    if total_images <= max_images:
        # 이미지가 3개 이하면 모두 선택
        images_to_analyze = image_urls
    else:
        # 첫 번째, 중간, 마지막 이미지 선택
        images_to_analyze = [
            image_urls[0],  # 첫 번째 이미지
            image_urls[total_images // 2],  # 중간 이미지
            image_urls[-1]  # 마지막 이미지
        ]
    
    # 여러 이미지를 하나의 메시지에 포함
    content = [
        {
            "type": "text", 
            "text": f"""다음 {len(images_to_analyze)}개의 쿠팡 상품 상세 이미지들을 분석해주세요.
모든 이미지에서 추출한 정보를 종합하여 하나의 JSON으로 정리해주세요.

추출할 정보:
- product_name: 제품명
- colors: 색상 옵션 (배열)
- sizes: 사이즈 정보 (배열)
- size_guide: 사이즈 가이드/치수표 (객체)
- material: 소재/재질
- manufacturer: 제조사/브랜드
- specifications: 세부 사양 (객체)
- features: 제품 특징 (배열)
- care_instructions: 관리/세탁 방법
- certifications: 인증 정보
- other_info: 기타 정보

없는 정보는 null로 표시하세요."""
        }
    ]
    
    # 각 이미지 URL 추가
    for url in images_to_analyze:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": url,
                "detail": "high"
            }
        })
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": content}],
        max_tokens=2000,
        timeout=30  # 타임아웃 설정 (30초)
    )
    
    return response.choices[0].message.content


print("\n📦 이미지 분석 함수 정의 완료!")
print("   - analyze_product_image(): 단일 이미지 분석")
print("   - analyze_multiple_images(): 여러 이미지 개별 분석")
print("   - analyze_product_images_combined(): 여러 이미지 통합 분석 (비용 효율적)")


✅ OpenAI API 키가 설정되었습니다.
   .env 파일 경로: c:\Users\주민우\Final-AI-Fork\.env

📦 이미지 분석 함수 정의 완료!
   - analyze_product_image(): 단일 이미지 분석
   - analyze_multiple_images(): 여러 이미지 개별 분석
   - analyze_product_images_combined(): 여러 이미지 통합 분석 (비용 효율적)


In [17]:
# Cell 4: 유틸리티 함수 정의

def clean_product_title(title):
    """
    상품명을 정제하여 불필요한 정보를 제거합니다.
    
    Args:
        title: 원본 상품명
        
    Returns:
        정제된 상품명
    """
    if not title:
        return ""
    
    # 줄바꿈으로 분리
    lines = title.split("\n")
    cleaned_lines = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # 글자가 전혀 없는(숫자/기호만 있는) 줄은 건너뛴다
        if not re.search(r'[가-힣a-zA-Z]', line):
            continue
        
        # 가격 패턴 제거 (숫자,원 포함)
        if re.search(r'[\d,]+\s*원', line):
            continue
        
        # 배송 관련 키워드 제거
        if any(keyword in line for keyword in ["도착", "배송", "무료배송", "로켓배송", "내일", "오늘"]):
            continue
        
        # 리뷰/평점 관련 제거
        if re.search(r'[\d.]+\s*\([\d,]+\)', line) or "리뷰" in line or "평점" in line:
            continue
        
        # 쿠폰/할인 관련 제거
        if any(keyword in line for keyword in ["쿠폰할인", "할인", "%", "적립", "캐시"]):
            continue
        
        # 기타 불필요한 키워드 제거
        if any(keyword in line for keyword in ["AD", "새 상품", "반품", "품절", "와우"]):
            continue
        
        cleaned_lines.append(line)
    
    # 첫 번째 줄만 사용 (보통 상품명)
    if cleaned_lines:
        return cleaned_lines[0]
    else:
        # 정제 후 비어있으면 원본 중 글자가 포함된 첫 줄을 사용
        for line in lines:
            candidate = line.strip()
            if candidate and re.search(r'[가-힣a-zA-Z]', candidate):
                return candidate
        if lines:
            return lines[0].strip()
    
    return ""


def extract_prices(item_elem):
    """
    상품 요소에서 원가와 할인가를 추출합니다.
    
    Args:
        item_elem: 상품 요소 (Selenium WebElement)
        
    Returns:
        tuple: (original_price, displayed_price)
    """
    original_price = ""
    displayed_price = ""
    
    try:
        # custom-oos 클래스를 가진 모든 요소 찾기
        all_custom_oos = item_elem.find_elements(By.CSS_SELECTOR, "[class*='custom-oos']")
        
        # 1단계: 원가 추출
        for elem in all_custom_oos:
            class_attr = elem.get_attribute("class") or ""
            tag_name = elem.tag_name.lower()
            price_text = elem.text.strip()
            
            if not price_text or "원" not in price_text:
                continue
            
            # 가격 숫자 추출
            price_match = re.search(r'([\d,]+)\s*원', price_text)
            if not price_match:
                price_match = re.search(r'([\d,]+)', price_text.replace(' ', ''))
            if not price_match:
                continue
            
            # 가격 값 추출 (쉼표 포함)
            price_value_raw = price_match.group(1)
            price_value = price_value_raw.replace(",", "")  # 숫자 비교용
            
            # 숫자가 실제로 있는지 확인 (쉼표만 있으면 제외)
            if not price_value or not price_value.isdigit():
                continue
            
            price_value_formatted = price_value_raw + "원"  # 원화 형식
            
            # 원가 판별: del 태그이거나 취소선이 있거나 작은 텍스트 크기
            is_original = False
            if tag_name == "del" or "fw-line-through" in class_attr:
                is_original = True
            elif "fw-text-[12px]" in class_attr or "fw-text-[14px]" in class_attr:
                # 작은 텍스트 크기 (12px, 14px)는 원가
                is_original = True
            
            # 원가 저장 (원가로 확실히 판별된 경우만)
            if is_original and not original_price:
                original_price = price_value_formatted
                break  # 첫 번째 원가만 저장
        
        # 2단계: 할인가 추출 (원가가 아니고, 큰 텍스트이거나 볼드인 경우만)
        for elem in all_custom_oos:
            class_attr = elem.get_attribute("class") or ""
            tag_name = elem.tag_name.lower()
            price_text = elem.text.strip()
            
            if not price_text or "원" not in price_text:
                continue
            
            # 가격 숫자 추출
            price_match = re.search(r'([\d,]+)\s*원', price_text)
            if not price_match:
                price_match = re.search(r'([\d,]+)', price_text.replace(' ', ''))
            if not price_match:
                continue
            
            # 가격 값 추출 (쉼표 포함)
            price_value_raw = price_match.group(1)
            price_value = price_value_raw.replace(",", "")  # 숫자 비교용
            
            # 숫자가 실제로 있는지 확인 (쉼표만 있으면 제외)
            if not price_value or not price_value.isdigit():
                continue
            
            price_value_formatted = price_value_raw + "원"  # 원화 형식
            
            # 이미 원가로 저장된 가격이면 스킵 (숫자만 비교)
            if original_price:
                original_price_num = original_price.replace(",", "").replace("원", "")
                if price_value == original_price_num:
                    continue
            
            # 원가 조건 체크 (del 태그, 취소선, 작은 텍스트는 제외)
            is_original = False
            if tag_name == "del":
                is_original = True
            elif "fw-line-through" in class_attr:
                is_original = True
            elif "fw-text-[12px]" in class_attr or "fw-text-[14px]" in class_attr:
                is_original = True
            
            # 원가가 아니고, 할인가 조건을 만족하는 경우만
            if not is_original:
                is_discount = False
                # 큰 텍스트 크기 (20px, 24px)는 할인가
                if "fw-text-[20px]" in class_attr or "fw-text-[24px]" in class_attr:
                    is_discount = True
                # 볼드이고 취소선이 없으면 할인가
                elif "fw-font-bold" in class_attr:
                    is_discount = True
                
                # 할인가 저장
                if is_discount and not displayed_price:
                    displayed_price = price_value_formatted
                    break  # 첫 번째 할인가만 저장
        
        # 할인가를 찾지 못했지만 원가는 있는 경우, 큰 텍스트 크기를 가진 요소 재검색
        if original_price and not displayed_price:
            for elem in all_custom_oos:
                class_attr = elem.get_attribute("class") or ""
                tag_name = elem.tag_name.lower()
                
                if tag_name == "del" or "fw-line-through" in class_attr:
                    continue
                
                # 큰 텍스트 크기나 볼드인 요소 찾기
                if ("fw-text-[20px]" in class_attr or 
                    "fw-text-[24px]" in class_attr or 
                    "fw-font-bold" in class_attr):
                    price_text = elem.text.strip()
                    if price_text and "원" in price_text:
                        price_match = re.search(r'([\d,]+)\s*원', price_text)
                        if not price_match:
                            price_match = re.search(r'([\d,]+)', price_text.replace(' ', ''))
                        if price_match:
                            price_value_raw = price_match.group(1)
                            price_value = price_value_raw.replace(",", "")
                            
                            # 숫자가 실제로 있는지 확인
                            if not price_value or not price_value.isdigit():
                                continue
                            
                            # 원가와 다른 가격만 할인가로 저장
                            original_price_num = original_price.replace(",", "").replace("원", "")
                            if price_value != original_price_num:
                                displayed_price = price_value_raw + "원"
                                break
        
        # 여전히 둘 다 비어 있으면 custom-oos에서 첫 가격을 displayed_price로 사용
        if not original_price and not displayed_price:
            for elem in all_custom_oos:
                price_text = elem.text.strip()
                if not price_text or "원" not in price_text:
                    continue
                price_match = re.search(r'([\d,]+)\s*원', price_text)
                if not price_match:
                    price_match = re.search(r'([\d,]+)', price_text.replace(' ', ''))
                if price_match:
                    price_value_raw = price_match.group(1)
                    price_value = price_value_raw.replace(",", "")
                    
                    # 숫자가 실제로 있는지 확인
                    if not price_value or not price_value.isdigit():
                        continue
                    
                    displayed_price = price_value_raw + "원"
                    break
    
    except Exception as e:
        pass
    
    return original_price, displayed_price


def extract_product_info(item_elem, idx):
    """
    상품 요소에서 상품 정보를 추출합니다.
    
    Args:
        item_elem: 상품 요소 (Selenium WebElement)
        idx: 상품 인덱스
        
    Returns:
        dict: 상품 정보 (title, original_price, displayed_price, product_link, thumbnail_url)
    """
    try:
        if idx > 0:
            time.sleep(0.5)
        
        # 상품명 추출
        title = ""
        title_selectors = [
            "a.ProductUnit_productName__P8nrl",
            "div.ProductUnit_productInfo__1l0il a",
            "div.ProductUnit_productInfo__1l0il",
            ".name",
            "[class*='name']"
        ]
        for sel in title_selectors:
            try:
                title_elem = item_elem.find_element(By.CSS_SELECTOR, sel)
                title = title_elem.text.strip()
                if title:
                    break
            except:
                continue
        
        # 상품명 정제
        title = clean_product_title(title)
        
        # 상품 링크 추출
        product_link = ""
        try:
            link_elem = item_elem.find_element(By.CSS_SELECTOR, "a")
            href = link_elem.get_attribute("href") or ""
            if href:
                if href.startswith("http"):
                    product_link = href
                elif href.startswith("/"):
                    product_link = f"https://www.coupang.com{href}"
        except:
            pass
        
        # 가격 정보 추출
        original_price, displayed_price = extract_prices(item_elem)
        
        # 썸네일 이미지 URL 추출
        thumbnail_url = ""
        try:
            img_elem = item_elem.find_element(By.CSS_SELECTOR, "img")
            thumbnail_url = img_elem.get_attribute("src") or ""
            if not thumbnail_url:
                thumbnail_url = img_elem.get_attribute("data-src") or ""
        except:
            pass
        
        # 최소한 제목이 있어야 유효한 상품
        if title:
            return {
                "title": title,
                "original_price": original_price,
                "displayed_price": displayed_price,
                "product_link": product_link,
                "thumbnail_url": thumbnail_url
            }
    
    except Exception as e:
        print(f"  상품 {idx+1} 정보 추출 오류: {e}")
    
    return None


def extract_detail_images(driver, product_link, idx):
    """
    상품 상세 페이지에서 상세 이미지 URL들을 추출합니다.
    
    Args:
        driver: Selenium WebDriver
        product_link: 상품 상세 페이지 URL
        idx: 상품 인덱스 (로깅용)
        
    Returns:
        list: 상세 이미지 URL 리스트
    """
    detail_images = []
    
    if not product_link:
        return detail_images
    
    try:
        print(f"  [{idx+1}] 상세 페이지 접속 중...")
        driver.get(product_link)
        time.sleep(3)  # 페이지 로드 대기
        
        # 상세 정보 섹션까지 스크롤
        try:
            driver.execute_script("window.scrollBy(0, window.innerHeight * 2)")
            time.sleep(2)
        except:
            pass
        
        # 방법 1: 제공된 CSS 클래스 구조를 이용한 추출
        # product-detail-content > product-detail-content-inside > vendor-item > type-HTML/type-TEXT > subType-TEXT > p > img
        selectors_to_try = [
            # 가장 구체적인 셀렉터부터 시도 (type-HTML과 type-TEXT 둘 다 포함)
            "div.product-detail-content-inside div.vendor-item div.type-HTML div.subType-TEXT p[style*='text-align:center'] img",
            "div.product-detail-content-inside div.vendor-item div.type-TEXT div.subType-TEXT p[style*='text-align:center'] img",
            "div.product-detail-content-inside div.vendor-item div[class*='type-HTML'] div.subType-TEXT p[style*='text-align:center'] img",
            "div.product-detail-content-inside div.vendor-item div[class*='type-TEXT'] div.subType-TEXT p[style*='text-align:center'] img",
            # text-align:center가 없는 p 태그도 포함
            "div.product-detail-content-inside div.vendor-item div.type-HTML div.subType-TEXT p img",
            "div.product-detail-content-inside div.vendor-item div.type-TEXT div.subType-TEXT p img",
            "div.product-detail-content-inside div.vendor-item div[class*='type-HTML'] div.subType-TEXT p img",
            "div.product-detail-content-inside div.vendor-item div[class*='type-TEXT'] div.subType-TEXT p img",
            # 더 넓은 범위
            "div.product-detail-content-inside div.vendor-item div[class*='type-'] p img",
            "div.product-detail-content-inside div.vendor-item p img",
            "div.product-detail-content-inside p img",
            # 클래스명 부분 매칭
            "div[class*='product-detail-content'] p[style*='text-align'] img",
            "div[class*='product-detail-content'] p img",
            "div[class*='vendor-item'] p img",
            "div[class*='subType-TEXT'] p img",
            # 더 넓은 범위
            "div.product-detail-content img",
            "div[class*='product-detail'] img"
        ]
        
        found_urls = set()  # 중복 제거용
        
        for selector in selectors_to_try:
            try:
                img_elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if img_elements:
                    for img in img_elements:
                        src = img.get_attribute("src") or ""
                        if not src:
                            src = img.get_attribute("data-src") or ""
                        
                        # 유효한 이미지 URL만 추가
                        if src and src.startswith("http") and src not in found_urls:
                            # 작은 아이콘이나 로고 제외 (일반적으로 상세 이미지는 큰 이미지)
                            # coupangcdn.com 도메인의 이미지만 포함
                            if "coupangcdn.com" in src or "coupang.com" in src:
                                found_urls.add(src)
                                detail_images.append(src)
                    
                    # 이미지를 찾았으면 계속 다른 셀렉터도 시도하여 더 많은 이미지 수집
                    # (break 제거하여 모든 셀렉터를 시도)
            except Exception as e:
                continue
        
        # 중복 제거 및 정렬
        if detail_images:
            # URL을 기준으로 중복 제거 (이미 found_urls로 처리했지만 안전을 위해)
            unique_images = []
            seen = set()
            for img_url in detail_images:
                if img_url not in seen:
                    seen.add(img_url)
                    unique_images.append(img_url)
            detail_images = unique_images
            print(f"      ✅ 상세 이미지 {len(detail_images)}개 발견")
        
        # 방법 2: 이미지가 없으면 더 넓은 범위에서 재시도
        if not detail_images:
            try:
                # 상품 상세 영역에서 모든 이미지 찾기
                all_detail_imgs = driver.find_elements(By.CSS_SELECTOR, "div[class*='product-detail'] img, div[class*='vendor'] img")
                for img in all_detail_imgs:
                    src = img.get_attribute("src") or img.get_attribute("data-src") or ""
                    if src and src.startswith("http") and src not in found_urls:
                        if "coupangcdn.com" in src or "coupang.com" in src:
                            # 썸네일이나 작은 이미지 제외 (URL에 특정 패턴이 있는 경우)
                            if "/vendor_inventory/" in src or "/image/" in src:
                                found_urls.add(src)
                                detail_images.append(src)
                
                if detail_images:
                    print(f"      ✅ 상세 이미지 {len(detail_images)}개 발견 (넓은 범위 검색)")
            except Exception as e:
                pass
        
        if not detail_images:
            print(f"      ⚠️ 상세 이미지를 찾을 수 없습니다.")
    
    except Exception as e:
        print(f"      ⚠️ 상세 페이지 접속 오류: {e}")
    
    return detail_images


def extract_from_markdown_code_block(text):
    """
    마크다운 코드 블록(```json ... ```)에서 내용을 추출합니다.
    
    Args:
        text: 원본 텍스트
        
    Returns:
        추출된 코드 블록 내용 또는 None
    """
    if "```" not in text:
        return None
    
    code_block_pattern = r'```(?:json)?\s*\n(.*?)\n```'
    match = re.search(code_block_pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def extract_json_by_brace_balance(text):
    """
    중괄호 균형을 체크하여 완전한 JSON 객체를 추출합니다.
    문자열 내부의 중괄호는 무시합니다.
    
    Args:
        text: JSON이 포함된 텍스트
        
    Returns:
        파싱된 JSON 객체 또는 None
    """
    brace_count = 0
    start_idx = -1
    in_string = False
    escape_next = False
    
    for i, char in enumerate(text):
        if escape_next:
            escape_next = False
            continue
        
        if char == '\\':
            escape_next = True
            continue
        
        if char == '"' and not escape_next:
            in_string = not in_string
            continue
        
        if not in_string:
            if char == '{':
                if start_idx == -1:
                    start_idx = i
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0 and start_idx != -1:
                    json_str = text[start_idx:i+1]
                    try:
                        return json.loads(json_str)
                    except json.JSONDecodeError:
                        # 다음 JSON 객체 시도
                        start_idx = -1
                        continue
    
    return None


def extract_json_from_text_start(text):
    """
    설명 텍스트를 제거하고 첫 번째 { 또는 [ 부터 시작하는 JSON을 추출합니다.
    
    Args:
        text: JSON이 포함된 텍스트
        
    Returns:
        파싱된 JSON 객체 또는 None
    """
    # 첫 번째 { 또는 [ 찾기
    json_start = -1
    for i, char in enumerate(text):
        if char in ['{', '[']:
            json_start = i
            break
    
    if json_start < 0:
        return None
    
    json_candidate = text[json_start:].strip()
    
    # 중괄호와 대괄호 균형 체크로 완전한 JSON 추출
    brace_count = 0
    bracket_count = 0
    in_string = False
    escape_next = False
    
    for i, char in enumerate(json_candidate):
        if escape_next:
            escape_next = False
            continue
        
        if char == '\\':
            escape_next = True
            continue
        
        if char == '"' and not escape_next:
            in_string = not in_string
            continue
        
        if not in_string:
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
            elif char == '[':
                bracket_count += 1
            elif char == ']':
                bracket_count -= 1
            
            # 중괄호와 대괄호가 모두 균형을 이룰 때
            if brace_count == 0 and bracket_count == 0 and i > 0:
                try:
                    return json.loads(json_candidate[:i+1])
                except json.JSONDecodeError:
                    pass
    
    # 전체 후보를 JSON으로 파싱 시도
    try:
        return json.loads(json_candidate)
    except json.JSONDecodeError:
        return None


def try_parse_as_json(text):
    """
    전체 텍스트를 JSON으로 직접 파싱을 시도합니다.
    
    Args:
        text: 파싱할 텍스트
        
    Returns:
        파싱된 JSON 객체 또는 None
    """
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def clean_and_return_text(text):
    """
    JSON 파싱에 실패한 경우 텍스트를 정제하여 반환합니다.
    
    Args:
        text: 정제할 텍스트
        
    Returns:
        정제된 텍스트
    """
    cleaned = text.replace("\\n", " ").replace("\\", "").replace("\n", " ").strip()
    # JSON 형식처럼 보이지만 파싱 실패한 경우 정제된 텍스트 반환
    if cleaned.startswith("{") or cleaned.startswith("["):
        return cleaned
    return text


def parse_extracted_info(extracted_info):
    """
    OpenAI 응답 문자열을 JSON/dict 형태로 변환합니다.
    마크다운 코드 블록이나 설명 텍스트가 포함된 경우에도 JSON을 추출합니다.
    
    여러 파싱 전략을 순차적으로 시도합니다:
    1. 마크다운 코드 블록에서 추출
    2. 중괄호 균형 체크로 JSON 추출
    3. 설명 텍스트 제거 후 JSON 추출
    4. 전체 텍스트를 JSON으로 직접 파싱
    5. 정제된 텍스트 반환
    
    Args:
        extracted_info: OpenAI 응답 (str, dict, 또는 기타 타입)
        
    Returns:
        파싱된 JSON 객체(dict) 또는 정제된 텍스트(str)
    """
    # 이미 dict면 그대로 반환
    if isinstance(extracted_info, dict):
        return extracted_info
    
    # 문자열이 아니면 그대로 반환
    if not isinstance(extracted_info, str):
        return extracted_info
    
    text = extracted_info.strip()
    
    # 전략 1: 마크다운 코드 블록에서 내용 추출
    code_content = extract_from_markdown_code_block(text)
    if code_content:
        text = code_content
    
    # 전략 2: 중괄호 균형 체크로 JSON 추출
    result = extract_json_by_brace_balance(text)
    if result is not None:
        return result
    
    # 전략 3: 설명 텍스트 제거 후 JSON 추출
    result = extract_json_from_text_start(text)
    if result is not None:
        return result
    
    # 전략 4: 전체 텍스트를 JSON으로 직접 파싱
    result = try_parse_as_json(text)
    if result is not None:
        return result
    
    # 전략 5: 정제된 텍스트 반환
    return clean_and_return_text(text)


In [18]:
# Cell 5: 크롤링 메인 함수

def crawl_coupang_products(
    search_url,
    max_products,
    headless=False,
    fetch_detail_images=True,
    max_pages=2,
):
    """
    쿠팡에서 상품을 크롤링합니다.
    
    Args:
        search_url: 검색 URL
        max_products: 최대 수집할 상품 개수
        headless: 헤드리스 모드 여부
        fetch_detail_images: 상세 이미지 추출 여부 (기본: True)
        max_pages: 검색 결과 페이지 탐색 최대 횟수
        
    Returns:
        list: 상품 정보 리스트
    """
    products = []
    driver = None
    seen_product_ids = set()
    seen_titles = set()
    
    try:
        # undetected-chromedriver 설정 (봇 감지 우회)
        options = uc.ChromeOptions()
        
        if headless:
            options.add_argument('--headless=new')
        
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--no-sandbox')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--start-maximized')
        
        # User-Agent 설정
        options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        
        print("브라우저 시작 중...")
        driver = uc.Chrome(options=options, version_main=None)
        
        print(f"접속 중: {search_url}")
        
        # 먼저 쿠팡 메인 페이지로 접속
        print("쿠팡 메인 페이지 접속 중...")
        driver.get("https://www.coupang.com/")
        time.sleep(3)
        
        if not headless:
            time.sleep(10)
        
        page_num = 1
        while len(products) < max_products and page_num <= max_pages:
            if page_num == 1:
                page_url = search_url
            else:
                separator = '&' if '?' in search_url else '?'
                page_url = f"{search_url}{separator}page={page_num}"
            
            # 검색 페이지로 이동
            print(f"\n검색 페이지로 이동 중 (페이지 {page_num}/{max_pages}): {page_url}")
            driver.get(page_url)
            
            # 페이지 로드 대기 (더 긴 대기 시간)
            print("페이지 로딩 대기 중...")
            if not headless:
                time.sleep(10)
            else:
                time.sleep(8)
            
            # 페이지 로딩 상태 확인
            try:
                WebDriverWait(driver, 10).until(
                    lambda d: d.execute_script("return document.readyState") == "complete"
                )
            except:
                print("⚠ 페이지 로딩 완료 대기 중 타임아웃")
            
            # 추가 대기 (동적 콘텐츠 로드를 위해)
            time.sleep(3)
            
            # Access Denied 체크 및 페이지 상태 확인
            page_title = driver.title
            current_url = driver.current_url
            
            print(f"📄 페이지 제목: {page_title if page_title else '(비어있음)'}")
            print(f"🔗 현재 URL: {current_url}")
            
            if page_title and "access" in page_title.lower():
                print("⚠ Access Denied 페이지가 감지되었습니다.")
                if not headless:
                    print("⚠ 브라우저 창에서 직접 새로고침을 시도해보세요.")
                    time.sleep(15)
                    page_title = driver.title
                    print(f"📄 새로고침 후 페이지 제목: {page_title}")
            
            if not page_title:
                print("⚠ 페이지 제목이 비어있습니다. 페이지가 제대로 로드되지 않았을 수 있습니다.")
                print("   잠시 더 대기한 후 재시도합니다...")
                time.sleep(5)
                page_title = driver.title
                if not page_title:
                    print("⚠ 여전히 페이지가 로드되지 않았습니다.")
                    print("   쿠팡 사이트가 봇을 차단했을 수 있습니다.")
            
            # 스크롤하여 동적 콘텐츠 로드
            print("상품 로딩 중...")
            for scroll_idx in range(5):
                driver.execute_script("window.scrollBy(0, window.innerHeight * 0.5)")
                time.sleep(2)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
            time.sleep(3)
            driver.execute_script("window.scrollTo(0, 0)")
            time.sleep(2)
            
            # 상품 리스트 찾기
            product_selectors = [
                "li[class*='ProductUnit_productUnit']",
                "li.ProductUnit_productUnit__Qd6sv",
                "ul#product-list li",
                "#productList li",
                "li[data-id]"
            ]
            
            product_items = []
            for selector in product_selectors:
                try:
                    items = driver.find_elements(By.CSS_SELECTOR, selector)
                    if items and len(items) > 0:
                        product_items = items
                        print(f"발견된 상품 수: {len(product_items)}개 (셀렉터: {selector})")
                        break
                except Exception:
                    continue
            
            if not product_items:
                print("\n기본 셀렉터로 찾지 못함. 넓은 범위로 재검색 중...")
                try:
                    all_lis = driver.find_elements(By.TAG_NAME, "li")
                    for li in all_lis:
                        try:
                            data_id = li.get_attribute("data-id")
                            class_name = li.get_attribute("class") or ""
                            if data_id or "ProductUnit" in class_name or "product" in class_name.lower():
                                product_items.append(li)
                        except Exception:
                            continue
                    if product_items:
                        print(f"넓은 범위 검색으로 {len(product_items)}개 요소 발견")
                except Exception as e:
                    print(f"넓은 범위 검색 중 오류: {e}")
            
            if not product_items:
                print("⚠ 상품 리스트를 찾을 수 없습니다.")
                print(f"현재 URL: {driver.current_url}")
                print(f"페이지 제목: {driver.title if driver.title else '(비어있음)'}")
                try:
                    page_source_length = len(driver.page_source)
                    print(f"📄 페이지 소스 길이: {page_source_length} 문자")
                    if "상품" in driver.page_source or "product" in driver.page_source.lower():
                        print("   ℹ️ 페이지에 '상품' 또는 'product' 키워드가 발견되었습니다.")
                    else:
                        print("   ⚠️ 페이지에 상품 관련 키워드가 없습니다.")
                    if "coupang" in driver.page_source.lower():
                        print("   ✅ 쿠팡 페이지로 확인됨")
                    else:
                        print("   ⚠️ 쿠팡 페이지가 아닐 수 있습니다.")
                except Exception as e:
                    print(f"   ⚠️ 페이지 소스 확인 중 오류: {e}")
            else:
                for idx, item_elem in enumerate(product_items):
                    if len(products) >= max_products:
                        break
                    product_data = extract_product_info(item_elem, idx)
                    if product_data:
                        product_link = product_data.get("product_link", "")
                        product_id = None
                        if product_link:
                            match = re.search(r'/products/(\d+)', product_link)
                            if match:
                                product_id = match.group(1)
                        is_duplicate = False
                        if product_id and product_id in seen_product_ids:
                            is_duplicate = True
                        elif product_data.get("title", "") in seen_titles:
                            is_duplicate = True
                        if not is_duplicate:
                            product_data["detail_images"] = []
                            products.append(product_data)
                            if product_id:
                                seen_product_ids.add(product_id)
                            if product_data.get("title"):
                                seen_titles.add(product_data.get("title"))
                        else:
                            print(f"  [{idx+1}] 중복 상품 스킵: {product_data.get('title', '')[:50]}...")
                        if len(products) >= max_products:
                            break
            
            print(f"➡ 페이지 {page_num} 처리 완료 (누적 {len(products)}개)")
            page_num += 1
            
        if not products:
            print("⚠ 어떤 상품도 수집하지 못했습니다.")
        
        # 상세 이미지 추출 (옵션이 활성화된 경우)
        if fetch_detail_images and products:
            print(f"\n📸 상세 이미지 추출 시작 ({len(products)}개 상품)...")
            print("   (각 상품 페이지에 접속하므로 시간이 걸릴 수 있습니다)\n")
            
            for idx, product in enumerate(products):
                product_link = product.get("product_link", "")
                if product_link:
                    detail_images = extract_detail_images(driver, product_link, idx)
                    product["detail_images"] = detail_images
                    
                    # 다음 상품 페이지로 이동하기 전 잠시 대기 (봇 감지 방지)
                    time.sleep(2)
            
            print(f"\n📸 상세 이미지 추출 완료!")
    
    except Exception as e:
        print(f"⚠ 크롤링 오류: {e}")
        import traceback
        traceback.print_exc()
    finally:
        try:
            if driver:
                driver.quit()
        except:
            pass
    
    return products


In [19]:
# Cell 6: 실행 함수 정의

def run_coupang_crawling(
    search_keyword,
    search_url,
    max_products,
    headless,
    fetch_detail_images,
    analyze_images,
    max_products_to_analyze,
    max_images_per_product,
    output_json
):
    """쿠팡 크롤링 + 이미지 분석 + JSON 저장을 실행합니다."""
    print("▶ 쿠팡 상품 검색 크롤링을 시작합니다 (undetected-chromedriver 사용)...\n")
    if fetch_detail_images:
        print("📸 상세 이미지 추출 기능이 활성화되어 있습니다.")
        print("   (비활성화하려면 fetch_detail_images=False 로 설정하세요)\n")

    products = crawl_coupang_products(
        search_url,
        max_products,
        headless=headless,
        fetch_detail_images=fetch_detail_images
    )

    total_detail_images = sum(len(p.get("detail_images", [])) for p in products)

    print(f"\n✅ 총 {len(products)}개 상품 정보 수집 완료")
    if fetch_detail_images:
        print(f"📸 총 {total_detail_images}개 상세 이미지 URL 수집 완료")

    if analyze_images and client:
        print(f"\n{'=' * 60}")
        print("🔍 상품 상세 이미지 분석을 시작합니다...")
        print(f"   분석할 상품 수: {min(max_products_to_analyze, len(products))}개")
        print(f"   상품당 이미지 수: 최대 {max_images_per_product}개\n")

        for idx, product in enumerate(products[:max_products_to_analyze]):
            product_title = product.get("title", "")[:40]
            detail_images = product.get("detail_images", [])

            print(f"\n📦 [{idx + 1}] {product_title}...")

            if not detail_images:
                print("   ⚠️ 상세 이미지가 없습니다.")
                product["extracted_info"] = None
                continue

            print(f"   🖼️ {len(detail_images)}개 이미지 중 {min(max_images_per_product, len(detail_images))}개 분석")

            extracted_info = analyze_product_images_combined(
                detail_images,
                max_images=max_images_per_product
            )

            parsed_info = parse_extracted_info(extracted_info)
            product["extracted_info"] = parsed_info
            print("   ✅ 분석 완료!")

            if isinstance(parsed_info, (dict, list)):
                preview_text = json.dumps(parsed_info, ensure_ascii=False)
            elif parsed_info is None:
                preview_text = "None"
            else:
                preview_text = str(parsed_info)

            if len(preview_text) > 200:
                print(f"   📝 미리보기: {preview_text[:200]}...")
            else:
                print(f"   📝 결과: {preview_text}")

        print(f"\n{'=' * 60}")
        print("✅ 이미지 분석 완료!")

    elif analyze_images and not client:
        print(f"\n⚠️ OpenAI 클라이언트가 초기화되지 않았습니다.")
        print("   OPENAI_API_KEY를 설정한 후 다시 실행해주세요.")
        print("   이미지 분석 없이 크롤링 결과만 저장합니다.")
    else:
        print("\nℹ️ 이미지 분석이 비활성화되어 있습니다.")
        print("   analyze_images=True 로 설정하여 활성화하세요.")

    result = {
        "search_keyword": search_keyword,
        "search_url": search_url,
        "total_products": len(products),
        "products": products
    }

    output_path = Path(output_json)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"\n✅ JSON 파일 저장 완료: {output_path}")
    if analyze_images and client:
        print("   (이미지 분석 결과 포함)")

    print(f"\n📋 수집된 상품 미리보기 (처음 5개):")
    for idx, product in enumerate(products[:5], 1):
        print(f"\n  {idx}. {product['title'][:60]}...")
        if product.get('original_price'):
            print(f"     원가: {product['original_price']}")
        if product.get('displayed_price'):
            print(f"     할인가격: {product['displayed_price']}")
        print(f"     링크: {product['product_link'][:70]}..." if product.get('product_link') else "     링크: 정보 없음")
        detail_imgs = product.get('detail_images', [])
        if detail_imgs:
            print(f"     상세 이미지: {len(detail_imgs)}개")
            print(f"       - {detail_imgs[0][:80]}...")
        else:
            print("     상세 이미지: 없음")
        if product.get('extracted_info'):
            print("     이미지 분석: ✅ 완료")
        elif analyze_images:
            print("     이미지 분석: ⏭️ 스킵됨")

    return result


In [21]:
# Cell 7: 실행 스크립트 생성 및 실행

FETCH_DETAIL_IMAGES = False
ANALYZE_IMAGES = True
MAX_PRODUCTS_TO_ANALYZE = 5
MAX_IMAGES_PER_PRODUCT = 3

print("▶ 쿠팡 상품 검색 크롤링을 시작합니다 (undetected-chromedriver 사용)...\n")
if FETCH_DETAIL_IMAGES:
    print("📸 상세 이미지 추출 기능이 활성화되어 있습니다.")
    print("   (비활성화하려면 FETCH_DETAIL_IMAGES = False 로 설정하세요)\n")

function_list = [
    retry_with_backoff,
    analyze_product_image,
    analyze_multiple_images,
    analyze_product_images_combined,
    clean_product_title,
    extract_prices,
    extract_product_info,
    extract_detail_images,
    extract_from_markdown_code_block,
    extract_json_by_brace_balance,
    extract_json_from_text_start,
    try_parse_as_json,
    clean_and_return_text,
    parse_extracted_info,
    crawl_coupang_products,
    run_coupang_crawling,
]

function_sources = [textwrap.dedent(inspect.getsource(fn)) for fn in function_list]
functions_code = "\n\n".join(function_sources)

project_root = Path.cwd().parent.parent
output_dir = Path.cwd()
prompt_literal = json.dumps(EXTRACTION_PROMPT, ensure_ascii=False)

script_lines = [
    "import os",
    "import json",
    "import time",
    "import re",
    "from pathlib import Path",
    "",
    "import undetected_chromedriver as uc",
    "from selenium.webdriver.common.by import By",
    "from selenium.webdriver.support.ui import WebDriverWait",
    "from selenium.webdriver.support import expected_conditions as EC",
    "from selenium.common.exceptions import TimeoutException, NoSuchElementException",
    "",
    "from dotenv import load_dotenv",
    "from openai import OpenAI, RateLimitError, APIError",
    "",
    f"PROJECT_ROOT = Path(r\"{str(project_root)}\")",
    f"OUTPUT_DIR = Path(r\"{str(output_dir)}\")",
    "ENV_PATH = PROJECT_ROOT / \".env\"",
    "load_dotenv(ENV_PATH)",
    "OPENAI_API_KEY = os.getenv(\"OPENAI_API_KEY\", \"\")",
    "client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None",
    f"EXTRACTION_PROMPT = {prompt_literal}",
    "",
    functions_code,
    "",
    "if __name__ == \"__main__\":",
    f"    result = run_coupang_crawling(",
    f"        search_keyword={SEARCH_KEYWORD!r},",
    f"        search_url={search_url!r},",
    f"        max_products={MAX_PRODUCTS},",
    f"        headless={NOTEBOOK_HEADLESS!r},",
    f"        fetch_detail_images={FETCH_DETAIL_IMAGES!r},",
    f"        analyze_images={ANALYZE_IMAGES!r},",
    f"        max_products_to_analyze={MAX_PRODUCTS_TO_ANALYZE},",
    f"        max_images_per_product={MAX_IMAGES_PER_PRODUCT},",
    "        output_json=str(OUTPUT_DIR / \"coupang_search_results.json\")",
    "    )",
    "    print(\"\\n✅ 크롤링 스크립트 실행 완료\")",
]
script_content = "\n".join(script_lines)

fd, script_path = tempfile.mkstemp(suffix="_crawl_coupang.py", text=True)
os.close(fd)
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_content)

try:
    completed = subprocess.run(
        [sys.executable, script_path],
        capture_output=True,
        text=True,
        check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(script_path)
    except Exception:
        pass


▶ 쿠팡 상품 검색 크롤링을 시작합니다 (undetected-chromedriver 사용)...

▶ 쿠팡 상품 검색 크롤링을 시작합니다 (undetected-chromedriver 사용)...

브라우저 시작 중...
접속 중: https://www.coupang.com/np/search?q=%EC%BB%B4%ED%93%A8%ED%84%B0
쿠팡 메인 페이지 접속 중...

검색 페이지로 이동 중 (페이지 1/2): https://www.coupang.com/np/search?q=%EC%BB%B4%ED%93%A8%ED%84%B0
페이지 로딩 대기 중...
📄 페이지 제목: 쿠팡이 추천하는 컴퓨터 관련 혜택과 특가
🔗 현재 URL: https://www.coupang.com/np/search?q=%EC%BB%B4%ED%93%A8%ED%84%B0
상품 로딩 중...
발견된 상품 수: 36개 (셀렉터: li[class*='ProductUnit_productUnit'])
  [4] 중복 상품 스킵: 조립 컴퓨터 조립PC 게이밍 고사양 게임용 본체 롤 오버워치 메이플 배틀그라운드 팰월드 디...
  [6] 중복 상품 스킵: 컴집 게이밍컴퓨터 컴퓨터본체 GTA6 배틀그라운드 디아블로4 롤 오버워치2 피파온라인 메이...
  [7] 중복 상품 스킵: 조립 컴퓨터 조립PC 게이밍 고사양 게임용 본체 롤 오버워치 메이플 배틀그라운드 팰월드 디...
  [8] 중복 상품 스킵: 피씨오브플레이어 컴퓨터 게이밍 조립컴퓨터 올인원 풀세트 모니터포함 고사양PC 오버워치 피파...
  [14] 중복 상품 스킵: 컴집 게이밍컴퓨터 컴퓨터본체 GTA6 배틀그라운드 디아블로4 롤 오버워치2 피파온라인 메이...
  [15] 중복 상품 스킵: 피씨오브플레이어 컴퓨터 게이밍 조립컴퓨터 올인원 풀세트 모니터포함 고사양PC 오버워치 피파...
  [22] 중복 상품 스킵: 보아스컴퓨터 게이밍PC 데스크탑 게임용 고사양 조립컴퓨터 조립PC 하이엔드PC 컴퓨터, W...
  [27] 중복 상품 스킵: